# Lab 1: VPG & PPO in PyTorch

**Course:** Deep Reinforcement Learning · **Framework:** PyTorch · **Estimated time:** ~60 min

## Learning Objectives

- Implement a minimal Vanilla Policy Gradient training loop from scratch and verify it learns CartPole-v1 within 50 epochs
- Extend VPG to PPO-Clip by adding the clipped surrogate objective, multiple gradient steps, and KL early stopping
- Run Spinning Up's full PPO on LunarLander-v2 and compare the effect of varying λ, network size, and the VPG baseline
- Design and execute a systematic ExperimentGrid sweep across 54 PPO configurations and interpret results
- Implement the diagonal Gaussian log-likelihood function (Problem Set 1.1)
- Implement an MLP diagonal Gaussian policy for PPO (Problem Set 1.2)
- Implement the TD3 critic and policy loss functions (Problem Set 1.3)

## Setup

Clone and install Spinning Up, then verify the installation.

In [ ]:
!git clone https://github.com/openai/spinningup.git
%cd spinningup
!pip install -e . -q

In [ ]:
# Verify: should print epoch-by-epoch AverageEpRet logs
!python -m spinup.run vpg_pytorch --env CartPole-v1 --epochs 5

## Exercise 1: Minimal VPG from Scratch

Implement a minimal VPG training loop **without using Spinning Up's VPG implementation** (use it only for reference).

Your implementation must:
1. Build a categorical policy network (for CartPole's discrete action space)
2. Collect one epoch of trajectories by rolling out the policy
3. Compute rewards-to-go for each timestep
4. Compute the pseudo-loss and take one gradient step

In [ ]:
import torch
import torch.nn as nn
from torch.distributions import Categorical
import gym

class PolicyNet(nn.Module):
    def __init__(self, obs_dim, act_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim, 64), nn.Tanh(),
            nn.Linear(64, 64),     nn.Tanh(),
            nn.Linear(64, act_dim)
        )

    def forward(self, obs):
        return Categorical(logits=self.net(obs))


def rewards_to_go(rewards):
    n = len(rewards)
    rtg = torch.zeros(n)
    running = 0
    for t in reversed(range(n)):
        running = rewards[t] + running  # no discount for simplicity
        rtg[t] = running
    return rtg


def collect_epoch(env, policy, steps=4000):
    obs_buf, act_buf, rtg_buf = [], [], []
    obs, _ = env.reset() if hasattr(env.reset(), '__len__') else (env.reset(), {})
    ep_rewards = []

    for _ in range(steps):
        obs_t = torch.as_tensor(obs, dtype=torch.float32)
        dist  = policy(obs_t)
        act   = dist.sample()

        obs_buf.append(obs)
        act_buf.append(act.item())

        result = env.step(act.item())
        obs, rew, done = result[0], result[1], result[2]
        ep_rewards.append(rew)

        if done:
            rtg_buf.extend(rewards_to_go(ep_rewards).tolist())
            result = env.reset()
            obs = result[0] if isinstance(result, tuple) else result
            ep_rewards = []

    return (
        torch.as_tensor(obs_buf, dtype=torch.float32),
        torch.as_tensor(act_buf, dtype=torch.int32),
        torch.as_tensor(rtg_buf, dtype=torch.float32),
    )


def train_vpg(env_name='CartPole-v1', epochs=50, steps=4000, lr=3e-4):
    env    = gym.make(env_name)
    policy = PolicyNet(env.observation_space.shape[0], env.action_space.n)
    optim  = torch.optim.Adam(policy.parameters(), lr=lr)

    for epoch in range(epochs):
        obs, acts, rtg = collect_epoch(env, policy, steps)
        optim.zero_grad()
        loss = -(policy(obs).log_prob(acts) * rtg).mean()
        loss.backward()
        optim.step()
        print(f'Epoch {epoch+1:3d} | mean_rtg={rtg.mean():.1f} | loss={loss.item():.4f}')

    return policy

policy = train_vpg()

**Tasks — try these modifications:**
- Add a value function network and subtract V(s) from RTG to get advantages. Compare learning curves: RTG weights vs. advantage weights.
- Does the advantage baseline speed up convergence?

## Exercise 2: Implement PPO-Clip

Extend your VPG implementation to PPO by adding:
1. Multiple gradient steps per epoch (`train_pi_iters`)
2. The clipped surrogate objective
3. Approximate KL early stopping

In [ ]:
def compute_ppo_loss(obs, acts, adv, logp_old, policy, clip_ratio=0.2):
    dist  = policy(obs)
    logp  = dist.log_prob(acts)
    ratio = torch.exp(logp - logp_old)

    clipped = torch.clamp(ratio, 1 - clip_ratio, 1 + clip_ratio)
    loss    = -torch.min(ratio * adv, clipped * adv).mean()

    approx_kl = (logp_old - logp).mean().item()
    return loss, approx_kl


def train_ppo_epoch(obs, acts, adv, logp_old, policy, optimizer,
                    clip_ratio=0.2, train_iters=80, target_kl=0.01):
    for i in range(train_iters):
        optimizer.zero_grad()
        loss, kl = compute_ppo_loss(obs, acts, adv, logp_old, policy, clip_ratio)
        if kl > 1.5 * target_kl:
            print(f'  KL early stop at step {i}, KL={kl:.4f}')
            break
        loss.backward()
        optimizer.step()
    return loss.item()


# Integrate with a value function baseline
class ValueNet(nn.Module):
    def __init__(self, obs_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim, 64), nn.Tanh(),
            nn.Linear(64, 64),     nn.Tanh(),
            nn.Linear(64, 1)
        )
    def forward(self, obs):
        return self.net(obs).squeeze(-1)


def train_ppo(env_name='CartPole-v1', epochs=50, steps=4000,
              pi_lr=3e-4, v_lr=1e-3, clip_ratio=0.2):
    env    = gym.make(env_name)
    policy = PolicyNet(env.observation_space.shape[0], env.action_space.n)
    vf     = ValueNet(env.observation_space.shape[0])
    pi_opt = torch.optim.Adam(policy.parameters(), lr=pi_lr)
    v_opt  = torch.optim.Adam(vf.parameters(), lr=v_lr)

    for epoch in range(epochs):
        obs, acts, rtg = collect_epoch(env, policy, steps)

        with torch.no_grad():
            vals    = vf(obs)
            adv     = rtg - vals
            adv     = (adv - adv.mean()) / (adv.std() + 1e-8)
            logp_old = policy(obs).log_prob(acts)

        # Policy update
        pi_loss = train_ppo_epoch(obs, acts, adv, logp_old, policy, pi_opt, clip_ratio)

        # Value update
        for _ in range(80):
            v_opt.zero_grad()
            v_loss = ((vf(obs) - rtg)**2).mean()
            v_loss.backward()
            v_opt.step()

        print(f'Epoch {epoch+1:3d} | mean_rtg={rtg.mean():.1f} | pi_loss={pi_loss:.4f} | v_loss={v_loss.item():.4f}')

train_ppo()

**Tasks:**
- Vary `clip_ratio` across `[0.1, 0.2, 0.3, 0.5]` and compare final performance.
- What happens when `clip_ratio=0.01`? When `clip_ratio=0.5`?

## Exercise 3: Run Spinning Up's PPO on LunarLander-v2

Use Spinning Up's full PPO (which includes GAE-Lambda, proper value function fitting, and structured logging) on a harder environment.

Run each experiment — the `--seed 0 10 20` flag will run 3 seeds automatically.

In [ ]:
# Baseline run
!python -m spinup.run ppo_pytorch \
    --env LunarLander-v2 \
    --hid "[64,64]" \
    --epochs 150 \
    --gamma 0.99 \
    --lam 0.97 \
    --clip_ratio 0.2 \
    --pi_lr 3e-4 \
    --vf_lr 1e-3 \
    --seed 0 10 20 \
    --exp_name ppo-lunar-baseline

In [ ]:
# Lower lambda (more bias, less variance)
!python -m spinup.run ppo_pytorch \
    --env LunarLander-v2 \
    --hid "[64,64]" \
    --epochs 150 \
    --lam 0.9 \
    --seed 0 10 20 \
    --exp_name ppo-lunar-lam09

In [ ]:
# Smaller architecture
!python -m spinup.run ppo_pytorch \
    --env LunarLander-v2 \
    --hid "[32,32]" \
    --epochs 150 \
    --seed 0 10 20 \
    --exp_name ppo-lunar-small

In [ ]:
# VPG baseline for comparison
!python -m spinup.run vpg_pytorch \
    --env LunarLander-v2 \
    --epochs 150 \
    --seed 0 10 20 \
    --exp_name vpg-lunar

In [ ]:
# Plot all runs together
!python -m spinup.run plot /root/spinningup/data/ppo-lunar-baseline \
                           /root/spinningup/data/ppo-lunar-lam09    \
                           /root/spinningup/data/ppo-lunar-small    \
                           /root/spinningup/data/vpg-lunar

**Analysis questions:**
- How many epochs until PPO reaches average return > 200 (considered "solved")?
- Does VPG converge at all on LunarLander within 150 epochs?
- Which `lam` value converges faster, and why?

## Exercise 4: ExperimentGrid Sweep

Use Spinning Up's `ExperimentGrid` to run a systematic hyperparameter search across 3 seeds × 3 architectures × 3 clip ratios × 2 λ values = **54 experiments**.

In [ ]:
import sys
sys.path.insert(0, '/content/spinningup')

from spinup.utils.run_utils import ExperimentGrid
from spinup import ppo_pytorch

eg = ExperimentGrid(name='ppo-lunar-sweep')
eg.add('env_name',             'LunarLander-v2',             '',    True)
eg.add('seed',                 [0, 10, 20])
eg.add('epochs',               100)
eg.add('ac_kwargs:hidden_sizes', [(32,32), (64,64), (128,128)], 'hid')
eg.add('clip_ratio',           [0.1, 0.2, 0.3],               'clip')
eg.add('lam',                  [0.95, 0.97],                   'lam')

eg.run(ppo_pytorch, num_cpu=1)

In [ ]:
!python -m spinup.run plot /root/spinningup/data/ppo-lunar-sweep

**Discussion:** From the sweep results, identify:
- Which architecture performed best on average?
- Is there an interaction between `clip_ratio` and `lam`?
- Which configuration has the lowest variance across seeds?

## Problem Set 1.1: Diagonal Gaussian Log-Likelihood

Implement a function that takes the means and log-stds of a batch of diagonal Gaussian distributions, along with previously-generated samples, and returns the log-likelihoods of those samples.

For a diagonal Gaussian with mean $\mu$ and diagonal covariance $\text{diag}(\sigma^2)$:

$$\log \pi(x|\mu, \sigma) = -\frac{1}{2} \sum_i \left( \frac{(x_i - \mu_i)^2}{\sigma_i^2} + 2 \log \sigma_i + \log 2\pi \right)$$

Open the exercise file and implement your solution there, then run the cell below to auto-verify against Spinning Up's reference implementation.

In [ ]:
# Inspect the exercise file
!cat /content/spinningup/spinup/exercises/pytorch/problem_set_1/exercise1_1.py

In [ ]:
# ── YOUR IMPLEMENTATION ──────────────────────────────────────────
# Edit exercise1_1.py directly, or implement here and paste it in.
#
# def gaussian_likelihood(x, mu, log_std):
#     ...
#     return total_log_prob   # shape: (batch,)
# ─────────────────────────────────────────────────────────────────

import numpy as np

def gaussian_likelihood(x, mu, log_std):
    """
    Args:
        x:       (batch, act_dim) — sampled actions
        mu:      (batch, act_dim) — distribution means
        log_std: (batch, act_dim) or (act_dim,) — log standard deviations
    Returns:
        log_prob: (batch,) — sum of per-dimension log-likelihoods
    """
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
# Auto-verify against Spinning Up's reference
!python /content/spinningup/spinup/exercises/pytorch/problem_set_1/exercise1_1.py

## Problem Set 1.2: MLP Diagonal Gaussian Policy for PPO

Implement an MLP diagonal Gaussian policy compatible with Spinning Up's PPO training loop.

Requirements:
- Accept observations and return a distribution supporting `.log_prob()` and `.sample()`
- Use the log-likelihood function from Exercise 1.1
- Use a standalone `nn.Parameter` for log-std (not a network output) for stability
- `log_prob` must return the **sum** of per-dimension log-likelihoods (scalar per sample, not a vector)

**Evaluation:** Your policy is tested by running 20 epochs on `InvertedPendulum-v2`. Success = average score > 500 in the last 5 epochs.

In [ ]:
!cat /content/spinningup/spinup/exercises/pytorch/problem_set_1/exercise1_2.py

In [ ]:
# ── YOUR IMPLEMENTATION ──────────────────────────────────────────
# Implement the MLPGaussianActor class in exercise1_2.py,
# then run the cell below to evaluate it.
# ─────────────────────────────────────────────────────────────────

In [ ]:
!python /content/spinningup/spinup/exercises/pytorch/problem_set_1/exercise1_2.py

## Problem Set 1.3: TD3 Computation Graph

Implement the TD3 critic and policy loss functions from starter code. You are given the full TD3 implementation except for the loss computations — find `# YOUR CODE HERE`.

**Critic update** (clipped double-Q with target smoothing):

$$y = r + \gamma (1-d) \min_{i=1,2} Q_{\phi_{\text{targ},i}}(s', \tilde{a}')$$
$$\mathcal{L}(\phi) = \mathbb{E}\left[(Q_\phi(s,a) - y)^2\right]$$

**Policy update** (delayed, every `policy_delay` steps):

$$\mathcal{L}(\theta) = -\mathbb{E}\left[Q_{\phi_1}(s, \mu_\theta(s))\right]$$

**Evaluation:** Within 10 epochs, HalfCheetah-v2 should exceed 300 and InvertedPendulum-v2 should reach 150.

In [ ]:
!cat /content/spinningup/spinup/exercises/pytorch/problem_set_1/exercise1_3.py

In [ ]:
# ── YOUR IMPLEMENTATION ──────────────────────────────────────────
# Fill in the loss functions in exercise1_3.py, then run below.
# ─────────────────────────────────────────────────────────────────

In [ ]:
!python /content/spinningup/spinup/exercises/pytorch/problem_set_1/exercise1_3.py \
    --env InvertedPendulum-v2

In [ ]:
!python /content/spinningup/spinup/exercises/pytorch/problem_set_1/exercise1_3.py \
    --env HalfCheetah-v2

In [ ]:
# Compare against reference solution
!python /content/spinningup/spinup/exercises/pytorch/problem_set_1/exercise1_3.py \
    --env HalfCheetah-v2 --use_soln